In [40]:
# Import necessary libraries
import os
from pathlib import Path
import numpy as np
import pandas as pd
from astropy import coordinates as coords
from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy.table import Table
from astropy.io import fits
from astropy.cosmology import LambdaCDM
from tqdm import tqdm
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

# Set cosmology
cosmo = LambdaCDM(H0=70, Om0=0.3, Ode0=0.7)

plt.rcParams.update({
    "font.family": 'STIXGeneral',
    'text.usetex': False,
    "mathtext.fontset": 'cm',
    "axes.labelweight": "bold",
    'font.size': 25,
    'font.weight': 'normal',
    
    # Tick direction and appearance
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top': True,            # show top ticks
    'ytick.right': True,          # show right ticks
    'xtick.minor.visible': True,  # show minor x ticks
    'ytick.minor.visible': True,  # show minor y ticks
    'xtick.major.size': 10,
    'xtick.minor.size': 6,
    'ytick.major.size': 10,
    'ytick.minor.size': 6,
    'xtick.major.width': 1.6,
    'xtick.minor.width': 1.6,
    'ytick.major.width': 1.6,
    'ytick.minor.width': 1.6,
    
    # Axes and line properties
    'lines.linewidth': 2,
    'axes.linewidth': 3.5,
    'axes.labelpad': 4,
    'xtick.major.pad': 7,
    'image.origin': 'lower'
})
# Pandas configuration
pd.set_option('display.max_columns', None)

In [41]:
df = pd.read_csv('./03e.A2199_mastercat_within35arcmin_raw1.csv')
df['p_modelmag_u_0'] = df['p_modelmag_u'] - df['p_extinction_u']
df['p_modelmag_g_0'] = df['p_modelmag_g'] - df['p_extinction_g']
df['p_modelmag_r_0'] = df['p_modelmag_r'] - df['p_extinction_r']
df['p_modelmag_i_0'] = df['p_modelmag_i'] - df['p_extinction_i']
df['p_modelmag_z_0'] = df['p_modelmag_z'] - df['p_extinction_z']

df['p_petromag_u_0'] = df['p_petromag_u'] - df['p_extinction_u']
df['p_petromag_g_0'] = df['p_petromag_g'] - df['p_extinction_g']
df['p_petromag_r_0'] = df['p_petromag_r'] - df['p_extinction_r']
df['p_petromag_i_0'] = df['p_petromag_i'] - df['p_extinction_i']  
df['p_petromag_z_0'] = df['p_petromag_z'] - df['p_extinction_z']  
df['grmod'] = df['p_modelmag_g_0'] - df['p_modelmag_r_0']

vis = pd.read_csv('./04d.A2199galaxy_visual_classification_result.csv')

In [53]:
# vis안에 들어가있는건 star-galaxy flag를 Strauss2002를 따라서 분류한것. 거기서 visualclasificaitno 한게 vis결과임
test = pd.merge(vis, df, on='p_objid', how='inner')
# vis와 inner merge 한 순간부터, galaxyflag는 들어갓다고 보면 됨.


In [54]:
# 등급 컷 적용. petro, r, corrected <= 21.0만 포함. 35 arcmin 이내만 포함

galaxy_mask = (test['p_probpsf']!=1)&(test['phot_source']!='wrong')&(test['grmod']>=-0.25)&(test['grmod']<=2.5)

test = test[(test['p_radgal']<=35)&(test['p_petromag_r_0']<=21.0)]

print(len(test))
print(len(test[test['galaxyflag']==0]))

4063
83


In [65]:
vis0 = vis[vis['galaxyflag']==1][['p_objid', 'galaxyflag']]

vis0 = pd.merge(vis0, df, on='p_objid', how='inner')

vis0 = vis0[vis0['z_tot_z']!=-9][['p_objid', 'galaxyflag', 'phot_source', 'p_ra', 'p_dec', 'p_petromag_r', 'p_modelmag_r', 'p_fibermag_r', 'p_extinction_r', 'z_tot_z', 'z_tot_zerr', 'z_tot_zsource', 'member']]

In [66]:
df[df['p_objid']==1237659330315485760][['p_objid', 'phot_source', 'p_ra', 'p_dec', 'p_petromag_r', 'p_modelmag_r', 'p_fibermag_r', 'p_extinction_r', 'z_tot_z', 'z_tot_zerr', 'z_tot_zsource', 'member']]

,p_objid,phot_source,p_ra,p_dec,p_petromag_r,p_modelmag_r,p_fibermag_r,p_extinction_r,z_tot_z,z_tot_zerr,z_tot_zsource,member
8042,1237659330315485760,DR9,247.275596,39.158823,19.99835,19.99934,21.65671,0.018663,0.030775,0.000195,mmt,Y


A2199 Members에 대해서 SDSS imglist 작업을 visual로 진행

1. petrosian magnitude 밝은 쪽에서 어두운쪽으로 order해서 0.1''/pixel로 이미지 관찰 
2. 주변 petrosian magnitude order에서 벗어난 (특히 더 어두워서) fiber magnitude 써야할 것 같은 녀석들 list 

In [67]:
gal_fibermag_a2199members_objid_list = [
    1237659330315223520, 
    1237659330315289013,
    1237659326566171345,
    1237659330852356638,
    1237659330852094660,
    # 1237659330315485821,
    # 1237659325492495126,
    # 1237659330315485760,
    # 1237655471820833890
]

비록 Redshift가 측정된 Target이라서 Galaxy라는 것이 이번 분석 과정에서 확실히 단정되었다고 하여도 그 Photometry가 Invalid하여 p_petromag_r을 사용하는 것이 신뢰할 수 없는 값인 경우가 있다. 이 경우에는 p_petromag_r을 p_fibermag_r로 교체해서 진행한다. 

아래의 objid list는 Redshift가 측정된 Target 중에서 SDSS imgliist로 visual classification을 진행해본 결과, photometry가 invalid한 candidate 후보군 들이다. (Nearby bright source, faint in image than fiber magnitude... 등이 있다.)

In [68]:
# gal_flag_2_withz_objid_list = [
#     1237659330315485209,
#     1237659326566171345,
#     1237659330315354281,
#     1237659325492299063,
#     1237659330852356638,
#     1237659326029365777,
#     1237659326566367268,
#     1237659326566301800,
#     1237659330315158338,
#     1237659326029430811,
#     1237659326566170816,
#     587733604804460611,
#     1237659330315354480,
#     1237659330315485821,
#     1237659325492363560,
#     1237659325492495126,
#     1237659330852225130,
#     1237659326566039624,
#     1237659325492494475,
#     1237659326566368038,
#     1237659326566236561,
#     1237659326029169873,
#     1237659326029300805,
#     1237659326029366691,
#     1237655471820833890,
#     1237655471820767366,
#     1237659326029562426,
#     1237659326566039622,
#     1237659326029430854,
#     1237659326029300990,
#     1237659325492363568,
#     1237659326566367925,
#     1237659326029366351
# ]

해당하는 녀석들 중에서 'MEMBER=Y' + petromag가 fibermag보다 1등급이상 밝은 녀석들을 골랐다. 이 녀석들은 nearby bright source에 의해서 실제로 어두운데 밝게 측정된 것이다. 즉, 주어진 SDSS img를 보았을 때 target galaxy의 magnitude가 petromag보다 더 어두워보여서 (주변 source의 오염에 의해 더 밝게 측정?) fiber magnitude를 활용하여 바꿔치기를 진행하고자 한다. 

In [69]:
vis_phot_update_target = vis0[(vis0['member']=='Y') & (vis0['p_objid'].isin(gal_fibermag_a2199members_objid_list))] # ((vis0['p_petromag_r']-vis0['p_fibermag_r'])<-1) 

In [70]:
vis_phot_update_target

,p_objid,galaxyflag,phot_source,p_ra,p_dec,p_petromag_r,p_modelmag_r,p_fibermag_r,p_extinction_r,z_tot_z,z_tot_zerr,z_tot_zsource,member
440,1237659330315223520,1,DR9,246.686151,39.408204,18.47701,18.43169,20.61015,0.030041,0.032082,0.000100,mmt,Y
504,1237659330315289013,1,DR9,246.783430,39.419026,18.70636,18.61250,20.60418,0.030149,0.030544,0.000140,mmt,Y
655,1237659326566171345,1,DR9,247.477661,39.831851,18.95364,18.95217,20.81891,0.028556,0.025114,0.000075,mmt,Y
769,1237659330852356638,1,DR9,247.594957,39.357696,19.28692,19.15230,21.45662,0.021666,0.031292,0.000166,mmt,Y
990,1237659330852094660,1,DR9,247.179917,39.848321,19.50634,19.45072,20.98326,0.033786,0.025254,0.000097,mmt,Y


# 2. Update galaxyflag column in df

In [71]:
# Merge df and vis on 'p_objid', keeping all rows from df and only 'galaxyflag' from vis
df = df.merge(vis[['p_objid', 'galaxyflag']], on='p_objid', how='left')

# Fill NaN values in galaxyflag with 0
df['galaxyflag'] = df['galaxyflag'].fillna(2).astype(int)

In [72]:
# df galaxyflag 0 이엇지만 z_tot_z가 잇어서 살아날 녀석들 
df[(df['galaxyflag']==0)&(df['z_tot_z']!=-9)]

,phot_source,p_phtype0,p_radgal,p_objid,p_ra,p_dec,p_petromag_u,p_petromagerr_u,p_petromag_g,p_petromagerr_g,p_petromag_r,p_petromagerr_r,p_petromag_i,p_petromagerr_i,p_petromag_z,p_petromagerr_z,p_modelmag_u,p_modelmagerr_u,p_modelmag_g,p_modelmagerr_g,p_modelmag_r,p_modelmagerr_r,p_modelmag_i,p_modelmagerr_i,p_modelmag_z,p_modelmagerr_z,p_fibermag_u,p_fibermagerr_u,p_fibermag_g,p_fibermagerr_g,p_fibermag_r,p_fibermagerr_r,p_fibermag_i,p_fibermagerr_i,p_fibermag_z,p_fibermagerr_z,p_extinction_u,p_extinction_g,p_extinction_r,p_extinction_i,p_extinction_z,p_petrorad_r,p_petroraderr_r,p_devrad_i,p_devab_i,p_run,p_rerun,p_camcol,p_field,p_efac,p_probpsf,z_mmt_xcr,z_mmt_tfilename,z_dfilename,z_filename,z_mmt_z,z_mmt_zerr,z_mmt_velqual,z_ned_name,z_ned_z,z_ned_zerr,z_sdss_z,z_sdss_zerr,z_desi_z,z_desi_zerr,z_tot_z,z_tot_zerr,z_tot_zsource,member,galaxyflag
6784,DR9,9,34.310827,1237659326566039624,247.164881,40.120524,20.92014,0.202990,20.51622,0.177419,20.26893,0.202888,20.22042,0.232829,20.46140,0.564343,20.78108,0.077803,20.40328,0.027540,20.1447,0.028153,20.14409,0.043723,20.31480,0.165597,20.54362,0.08160,19.74728,0.046923,19.40042,0.048659,19.20403,0.048865,19.12719,0.089858,0.046811,0.034443,0.024981,0.018942,0.013430,1.411499,0.078736,0.602768,0.631269,3225,301,5,235,0.868504,0,-9.0,NN,NN,nn.fits,-9.0,-9.0,N,SDSS J162839.48+400715.3,0.02573,-9.0,-9.0,-9.0,-9.0,-9.0,0.02573,-9.0,ned,N,0
6825,DR9,9,34.504799,1237659326566039622,247.167775,40.123733,21.28598,0.248934,20.74809,0.170312,20.85733,0.209165,20.86362,0.265536,20.54635,0.432475,21.13147,0.096742,20.63577,0.031788,20.7278,0.042953,20.74673,0.065837,20.37562,0.155065,20.50217,0.08082,19.63951,0.043076,19.23929,0.030158,19.02013,0.031225,18.87672,0.060586,0.046794,0.034430,0.024972,0.018935,0.013425,1.201580,0.089961,0.245595,0.880606,3225,301,5,235,1.618038,0,-9.0,NN,NN,nn.fits,-9.0,-9.0,N,SDSS J162840.25+400725.2,0.85500,-9.0,-9.0,-9.0,-9.0,-9.0,0.85500,-9.0,ned,N,0


In [73]:
# Case 1: galaxyflag == 2 and z_tot_z != -9 → set galaxyflag = 1
mask1 = (df['galaxyflag'] == 2) & (df['z_tot_z'] != -9)
df.loc[mask1, 'galaxyflag'] = 1

# Case 2: galaxyflag == 2 and z_tot_z == -9 → set galaxyflag = 0
mask2 = (df['galaxyflag'] == 2) & (df['z_tot_z'] == -9)
df.loc[mask2, 'galaxyflag'] = 0


# 3. Update photometry put $r_\mathrm{fiber}$ into $r_\mathrm{petro}$ for "vis_phot_update_target" galaxies 

In [74]:
import pandas as pd

# Create a set of target objids for fast lookup
target_objids = set(vis_phot_update_target['p_objid'])

# Make a copy of the original DataFrame
df_new = df.copy()

# Create a mask for rows to update
update_mask = df_new['p_objid'].isin(target_objids)

# Apply the update
df_new.loc[update_mask, 'p_petromag_r'] = df_new.loc[update_mask, 'p_fibermag_r']

# Create the 'photflag' column
df_new['photflag'] = 0
df_new.loc[update_mask, 'photflag'] = 1

In [75]:
df_new.to_csv('04f.A2199_mastercat_within35arcmin_flag_update.csv', index=False)